# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hassaan-Raza/FlyRank-Intership/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

**Finding 1 - ML Appendix, Feature Importance (Random Forest predicting Health Score):**
 The paper reports Average Position as the top predictor at 43% importance,
followed by Impressions (32%) and Scroll Depth (15%).

Where does the label come from: Health Score is explicitly defined earlier in
the paper as Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll
Depth (20 pts). Three of the model's top four "important" features are literal
components of the target it's predicting. The paper does flag this honestly
in its own caveat ("high importance is therefore expected and does not imply
external causation"), which is good practice.

Does the validation design carry the claim: not fully, since no split can fix
a target that's partly built from the inputs being ranked. This is the same
circularity I had to catch in my own Week 4 baseline, where a rule scored
1.00 Precision@50 only because it directly encoded the label into its scoring
logic. A cleaner test would predict a genuinely external outcome instead of a
composite score that already contains the features being ranked.

**Finding 2 - AI Model Performance (OpenAI vs. Gemini, age-controlled cohorts):**
The paper concludes that once age mix is controlled, "Gemini leads some
cohorts and OpenAI leads others... output quality, editing, topic fit, and
rollout timing matter more than a simple AI-versus-human framing."

Where does the label come from: the paper doesn't state how "which model
wrote this page" was captured, self-reported metadata, a classifier, a
naming convention, across 57 brands and 5 models. That's a real gap.

Does the validation design carry the claim: age-controlling the cohorts is
good practice, but it only helps if the underlying model-attribution label is
reliable in the first place. If that label is noisy, the careful age-matching
built on top of it inherits the same noise. This mirrors a question I had to
ask of my own trend_direction label in Week 1, it's a proxy, and I only
trusted it once I confirmed exactly how it was computed.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

**Before/after:** The naive random split scored AUC=0.776, AP=0.793, noticeably
higher than the client-grouped split's AUC=0.607, AP=0.599. That ~17-point
AUC gap is the leakage signature: without grouping, the model partly
memorized client-specific patterns (rows from the same client appeared in
both train and test), inflating its apparent performance. The grouped
number is the honest one to report and use going forward, even though it's
lower, it reflects how the model would actually perform on a genuinely new
client it hasn't seen before, which is the real-world use case.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

# Reload the same data and features as Week 5
df = pd.read_csv("content_refresh_anonymized.csv")
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

feature_cols = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'engaged_sessions_90d',
    'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions',
    'days_with_sessions', 'content_age_days', 'days_since_last_update',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct'
]

X = df[feature_cols].fillna(0)
y = df['is_declining']
groups = df['client_id']

# BEFORE: naive random split, no client grouping
X_train_naive, X_test_naive, y_train_naive, y_test_naive = train_test_split(
    X, y, test_size=0.2, random_state=42
)
rf_naive = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced').fit(X_train_naive, y_train_naive)
naive_probs = rf_naive.predict_proba(X_test_naive)[:, 1]
naive_auc = roc_auc_score(y_test_naive, naive_probs)
naive_ap = average_precision_score(y_test_naive, naive_probs)

# AFTER: client-grouped split (same approach as Week 5)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train_grp, X_test_grp = X.iloc[train_idx], X.iloc[test_idx]
y_train_grp, y_test_grp = y.iloc[train_idx], y.iloc[test_idx]

rf_grouped = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced').fit(X_train_grp, y_train_grp)
grouped_probs = rf_grouped.predict_proba(X_test_grp)[:, 1]
grouped_auc = roc_auc_score(y_test_grp, grouped_probs)
grouped_ap = average_precision_score(y_test_grp, grouped_probs)

print(f"BEFORE (naive random split): AUC={naive_auc:.3f}, AP={naive_ap:.3f}")
print(f"AFTER (client-grouped split): AUC={grouped_auc:.3f}, AP={grouped_ap:.3f}")

BEFORE (naive random split): AUC=0.776, AP=0.793
AFTER (client-grouped split): AUC=0.607, AP=0.599


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
excluded_cols = ['trend_pct', 'impressions_last_30d', 'impressions_prev_30d',
                  'clicks_last_30d', 'clicks_prev_30d', 'sessions_last_30d', 'sessions_prev_30d']
product_flags = ['health_score', 'priority_score', 'action_type']

leaked = [c for c in excluded_cols + product_flags if c in feature_cols]
print(f"Leakage check on final feature set: {'FAIL - leaked columns found: ' + str(leaked) if leaked else 'PASS - no excluded columns present'}")
print(f"\nFinal feature set used: {feature_cols}")

Leakage check on final feature set: PASS - no excluded columns present

Final feature set used: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']


## 4. Claim rewrite

Random Forest and Logistic Regression both showed
modestly higher ROC AUC and Average Precision than random chance on a
client-grouped holdout. The Week 4 baseline's Precision@50 was not directly
comparable, its scoring rule only assigns nonzero scores to rows that already
match the label, so its 1.00 Precision@50 reflects that circularity rather
than real predictive skill. Of the model's top 50 ranked candidates, 20 fell
outside the baseline's scoring gate entirely; of those, 1 was a genuinely
declining page, a modest, directional signal, not a dramatic one.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.